# Linear Programming

Overarching objective: Minimize or maximize a desirable variable given a set of constraints. 

This might be best if we were to go through this step by step.

Maximise: `Z = 5x₁ + 8x₂`

Constraints:
```
2x₁ + 4x₂ ≤ 40   (wood hours)
3x₁ + 2x₂ ≤ 36   (labour hours)
x₁ ≥ 0,  x₂ ≥ 0
```

In [19]:
import numpy as np

# Objective function coefficients (we maximise, so these are positive)
c = np.array([5.0, 8.0])

# Constraint left-hand side (one row per constraint)
A = np.array([
    [2.0, 4.0],
    [3.0, 2.0]
])

# Constraint right-hand side (the limits)
b = np.array([40.0, 36.0])

The simplex method needs equalities, not inequalities. So we would convert each inequality-structured constraint by adding a slack variable that soaks up the difference.

```
2x₁ + 4x₂ ≤ 40   →   2x₁ + 4x₂ + s₁ = 40
3x₁ + 2x₂ ≤ 36   →   3x₁ + 2x₂ + s₂ = 36
```

s₁ and s₂ represent unused resources — if you use 30 of your 40 wood hours, then s₁ = 10. When a constraint is tight (fully used), its slack variable is 0.

In [20]:
def build_tableau(c, A, b):
    num_constraints, num_variables = A.shape
    print(f"Number of constraints: {num_constraints}\n Number of variables: {num_variables}")

    # Add an identity matrix to the right of A (one slack variable per constraint)
    slack = np.eye(num_constraints)
    # print(f"Slack: {slack}")

    # Stack A and slack side by side → shape: (num_constraints, num_vars + num_constraints)
    tableau = np.hstack([A, slack])

    # Add the b column on the far right
    tableau = np.hstack([tableau, b.reshape(-1, 1)])

    # Build the objective row:
    # Negate c because we track -Z and drive it toward zero
    # Slack variables and b column get zeros in the objective row
    obj_row = np.hstack([-c, np.zeros(num_constraints + 1)])

    # Stack objective row below the constraint rows
    tableau = np.vstack([tableau, obj_row])
    # print(f"Tableau shape: {tableau.shape}")


    return tableau

tableau = build_tableau(c, A, b)
print(tableau)

Number of constraints: 2
 Number of variables: 2
[[ 2.  4.  1.  0. 40.]
 [ 3.  2.  0.  1. 36.]
 [-5. -8.  0.  0.  0.]]


Remember the objective row is currently [-5, -8, 0, 0, 0]. The negative values mean "if you increase this variable, Z improves." We want to pick the most negative value — that's the variable that improves Z the fastest.

This is called the entering variable — it's about to enter the solution.

In [21]:
def get_pivot_column(tableau):
    # Look only at the objective row (last row), excluding the RHS column (last column)
    obj_row = tableau[-1, :-1]
    # print(f"Objective row: {obj_row}")
    
    # Find the index of the most negative value
    col = np.argmin(obj_row)
    
    # If nothing is negative, we're already optimal — no pivot needed
    if obj_row[col] >= 0:
        return None
    
    return col

pivot_col = get_pivot_column(tableau)
print(f"Pivot column: {pivot_col}")  # → 1  (x₂, since -8 is most negative)

Pivot column: 1


In [22]:
def get_pivot_row(tableau, pivot_col):
    # RHS is the last column
    rhs = tableau[:-1, -1]
    
    # Values in the pivot column (excluding the objective row)
    col_vals = tableau[:-1, pivot_col]
    # print(f"Values in the pivot column: {col_vals}")
    
    # We can only divide by positive values
    # A zero or negative value in the pivot column means that constraint
    # doesn't actually limit how far we can go — so we mask those out
    ratios = np.where(col_vals > 0, rhs / col_vals, np.inf)
    
    # The row with the smallest ratio is our pivot row
    row = np.argmin(ratios)
    
    # If every ratio is infinite, the problem is unbounded (no ceiling exists)
    if ratios[row] == np.inf:
        raise ValueError("Problem is unbounded — Z can grow forever")
    
    return row

pivot_row = get_pivot_row(tableau, pivot_col)
print(f"Pivot row: {pivot_row}")  # → 0  (wood constraint is binding)

Pivot row: 0


In [23]:
def pivot(tableau, pivot_row, pivot_col):
    # Work on a copy so we don't mutate the original
    t = tableau.copy()
    
    # Step 1 — normalise the pivot row
    # Divide the entire pivot row by the pivot element
    # This makes the pivot element exactly 1
    pivot_element = t[pivot_row, pivot_col]
    t[pivot_row] = t[pivot_row] / pivot_element

    # Step 2 — eliminate all other entries in the pivot column
    # For every other row, subtract a multiple of the pivot row
    # so that their pivot column entry becomes 0
    for i in range(len(t)):
        if i == pivot_row:
            continue  # skip the pivot row itself
        factor = t[i, pivot_col]
        t[i] = t[i] - factor * t[pivot_row]

    return t

tableau = pivot(tableau, pivot_row, pivot_col)
print(tableau)

[[ 0.5   1.    0.25  0.   10.  ]
 [ 2.    0.   -0.5   1.   16.  ]
 [-1.    0.    2.    0.   80.  ]]


In [24]:
def simplex(c, A, b):
    tableau = build_tableau(c, A, b)
    num_constraints, num_vars = A.shape

    print("Initial tableau:")
    print(tableau, "\n")

    iteration = 0

    while True:
        iteration += 1

        # Step 3 — find the pivot column
        pivot_col = get_pivot_column(tableau)

        # If no negative values in the objective row, we're optimal
        if pivot_col is None:
            print(f"Optimal solution found after {iteration - 1} iterations!\n")
            break

        # Step 4 — find the pivot row
        pivot_row = get_pivot_row(tableau, pivot_col)

        print(f"Iteration {iteration}: pivot on row {pivot_row}, col {pivot_col}")

        # Step 5 — perform the pivot
        tableau = pivot(tableau, pivot_row, pivot_col)
        print(tableau, "\n")

    # Extract the solution
    # For each original variable, check if its column is a unit vector
    # If it is, the variable's value is the RHS of the row where the 1 sits
    # If it isn't, the variable is non-basic (not in the solution) → value is 0
    solution = np.zeros(num_vars)
    for j in range(num_vars):
        col = tableau[:-1, j]
        # A unit vector has exactly one 1 and the rest 0s
        if np.sum(col == 1) == 1 and np.sum(col == 0) == num_constraints - 1:
            row = np.argmax(col)
            solution[j] = tableau[row, -1]

    optimal_value = tableau[-1, -1]

    print(f"x₁ (chairs) = {solution[0]}")
    print(f"x₂ (tables) = {solution[1]}")
    print(f"Maximum Z   = {optimal_value}")

    return solution, optimal_value

# Run it!
solution, Z = simplex(c, A, b)

Number of constraints: 2
 Number of variables: 2
Initial tableau:
[[ 2.  4.  1.  0. 40.]
 [ 3.  2.  0.  1. 36.]
 [-5. -8.  0.  0.  0.]] 

Iteration 1: pivot on row 0, col 1
[[ 0.5   1.    0.25  0.   10.  ]
 [ 2.    0.   -0.5   1.   16.  ]
 [-1.    0.    2.    0.   80.  ]] 

Iteration 2: pivot on row 1, col 0
[[ 0.     1.     0.375 -0.25   6.   ]
 [ 1.     0.    -0.25   0.5    8.   ]
 [ 0.     0.     1.75   0.5   88.   ]] 

Optimal solution found after 2 iterations!

x₁ (chairs) = 8.0
x₂ (tables) = 6.0
Maximum Z   = 88.0
